<a href="https://colab.research.google.com/github/victorialms/Exec3LimpezaPandas/blob/main/atividade_pratica_aula3_limpeza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas

Esta atividade foi construída com base nos slides da Aula 3, que destacam a **limpeza como a fundação invisível** de qualquer dashboard confiável. O objetivo aqui não é apenas "fazer funcionar", mas tomar decisões conscientes sobre tipos, valores ausentes, duplicidades, variáveis derivadas e exportação da base limpa. fileciteturn2file0

## Regras desta atividade
- Você deve **construir os códigos**.
- O notebook orienta os passos, mas não entrega as soluções prontas.
- Sempre que fizer uma decisão de limpeza, **documente o porquê** em comentário ou célula markdown.
- Ao final, exporte uma base limpa para uso nas próximas aulas.

## Dataset desta atividade
Arquivo bruto: `vendas_brasil_aula3_bruto.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para trabalhar com:
- leitura de dados
- tratamento de nulos
- identificação de infinitos
- exportação do resultado final

**Sugestão:** Pandas e NumPy já resolvem toda a atividade.


In [ ]:
# Escreva aqui suas importações

import pandas as pd
import numpy as np


## 2. Leitura da base bruta

Leia o arquivo `vendas_brasil_aula3_bruto.csv` em um DataFrame chamado `df`.

Depois:
1. Exiba as primeiras linhas
2. Exiba as últimas linhas
3. Observe visualmente se já existem sinais de sujeira


In [ ]:
# Leia o CSV e faça uma inspeção inicial

df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

# primeiras e ultimas linhas (1 e 2)
df.head()
df.tail()


,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao
223,1180,2024-03-11,SC,Loja Física,Telefonia,Smartphone X,C158,1,2563.44,415.42,NaN
224,1015,2024-02-26,RS,Online,Informática,Notebook Pro,C102,3,15011.49,NaN,cliente premium
225,1115,2024-07-29,MG,Loja Física,Informática,Notebook Pro,C144,3,12566.04,2879.76,NaN
226,1172,2024-11-11,ES,Loja Física,Informática,Notebook Pro,C128,3,13154.04,3408.22,entrega rápida
227,1209,2024-05-27,MG,Marketplace,Áudio,Caixa de Som,C121,2,810.2,282.52,NaN


###Resposta 3:

###### Observando visualmente, já é possível identificar possíveis problemas como: valores estranhos, formatos diferentes e possíveis inconsistências nos dados.

## 3. Check-up inicial do dataset

Com base no checklist de um dataset "clean" apresentado na aula, faça um diagnóstico inicial da base. fileciteturn2file0

### Sua missão
Use comandos que permitam responder:
- Qual é o tamanho da base?
- Quais são os tipos atuais das colunas?
- Existem valores nulos?
- Há colunas com tipo inadequado?
- Há algo que pareça impedir análises confiáveis?


# Respostas:

### Tamanho da base:
##### A base possui 228 linhas e 11 colunas.

### Tipos das colunas:
##### Existem muitas colunas como object (texto), como data, uf, canal, categoria, produto, cliente_id, receita e observacao.

### Valores nulos:
##### Foram encontrados valores nulos nas colunas: uf (5), canal (5), categoria (5), receita (2), lucro (6) e observacao (50).

### Tipos inadequados:
##### A coluna 'data' está como object, mas deveria ser datetime. A coluna 'receita' também está como object, mas deveria ser numérica.

### Problemas que podem impedir análise:
##### A presença de valores nulos em colunas importantes como receita e lucro pode afetar os resultados. Além disso, tipos incorretos (como data e receita como texto) podem gerar erros em análises e cálculos.

In [ ]:
# Faça aqui o check-up inicial: shape, info, dtypes, isna, etc.

df.shape
df.info()
df.dtypes
df.isna().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pedido_id   228 non-null    int64  
 1   data        228 non-null    object 
 2   uf          223 non-null    object 
 3   canal       223 non-null    object 
 4   categoria   223 non-null    object 
 5   produto     228 non-null    object 
 6   cliente_id  228 non-null    object 
 7   quantidade  228 non-null    int64  
 8   receita     226 non-null    object 
 9   lucro       222 non-null    float64
 10  observacao  178 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 19.7+ KB


,0
pedido_id,0
data,0
uf,5
canal,5
categoria,5
produto,0
cliente_id,0
quantidade,0
receita,2
lucro,6


## 4. Batalha 1 — A tirania dos tipos de dados

Nos slides, vimos que datas não podem ser tratadas como texto e que números em formato string quebram análises. fileciteturn2file0

### Tarefa
Converta corretamente, quando fizer sentido:
- `data`
- `receita`
- `lucro`
- `quantidade`

### Orientação
- Investigue valores estranhos antes de converter
- Pense no uso de `errors='coerce'`
- Registre em markdown ou comentário quais problemas encontrou


In [ ]:
# Explore valores problemáticos e faça as conversões de tipo

# 1.
print("Exemplos de valores na coluna 'receita' antes de limpar:")
print(df['receita'].unique()[:5])

# 2.
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# 3.
if df['receita'].dtype == 'object':
    # Removemos possíveis espaços em branco nas pontas
    df['receita'] = df['receita'].str.strip()
    # Substituímos vírgula por ponto para o padrão americano (float)
    df['receita'] = df['receita'].str.replace(',', '.', regex=False)
    # Convertemos para numérico. O 'coerce' transforma o que não for número em NaN
    df['receita'] = pd.to_numeric(df['receita'], errors='coerce')

# 4.
df['lucro'] = pd.to_numeric(df['lucro'], errors='coerce')
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

# 5.
print("\n--- Tipos após a conversão ---")
print(df.dtypes[['data', 'receita', 'lucro', 'quantidade']])
print("\n--- Valores Nulos gerados por erro de conversão ---")
print(df[['data', 'receita', 'lucro', 'quantidade']].isna().sum())


Exemplos de valores na coluna 'receita' antes de limpar:
[ 4535.11  4005.59   309.02  7943.78 19926.65]

--- Tipos após a conversão ---
data          datetime64[ns]
receita              float64
lucro                float64
quantidade             int64
dtype: object

--- Valores Nulos gerados por erro de conversão ---
data          224
receita        15
lucro           6
quantidade      0
dtype: int64


### Reflexão curta
Explique:
1. Por que deixar `data` como texto pode quebrar análises temporais?
2. Por que deixar `receita` como string pode distorcer cálculos?

##Resposta:

##### 1. O computador entende texto apenas como uma sequência de caracteres. Se a data for texto, o Python colocará "10/01/2024" antes de "02/01/2024" (ordem alfabética pelo primeiro dígito), o que destrói qualquer gráfico de linha ou cálculo de sazonalidade. Como datetime, o Pandas entende a cronologia e permite extrair automaticamente o dia da semana, mês ou ano.

##### 2. Strings não aceitam operações matemáticas como soma ou média. Se você tentar somar duas strings "100" + "200", o resultado será "100200" (concatenação) em vez de 300. Além disso, strings tratam vírgulas e pontos apenas como símbolos, impedindo qualquer análise estatística ou financeira confiável.

## 5. Batalha 2 — O enigma dos valores ausentes

A aula reforça que valores ausentes não devem ser tratados automaticamente; a decisão precisa ser justificada. fileciteturn2file0

### Tarefa
1. Mapeie os valores ausentes por coluna
2. Identifique quais colunas críticas têm nulos
3. Defina uma estratégia para cada caso:
   - remover linhas?
   - preencher?
   - manter?
4. Justifique cada escolha

### Dica
Evite decisões automáticas sem análise de contexto.


In [ ]:
# Investigue os nulos e aplique aqui sua estratégia de tratamento

display(df[df['receita'].isna()])

df.dropna(subset=['receita'], inplace=True)

df['uf'] = df['uf'].fillna('Não Informado')
df['canal'] = df['canal'].fillna('Não Informado')
df['categoria'] = df['categoria'].fillna('Não Informado')

df['lucro'] = df['lucro'].fillna(df['lucro'].median())

df['observacao'] = df['observacao'].fillna('Sem Observação')

print(df.isna().sum())


,pedido_id,data,uf,canal,categoria,produto,cliente_id,quantidade,receita,lucro,observacao
14,1014,NaT,RS,Loja Física,Periféricos,Teclado Mecânico,C159,2,NaN,323.62,ok
17,1017,2024-09-30,SP,Marketplace,Periféricos,Mouse Gamer,C145,2,NaN,237.40,ok
32,1032,NaT,PR,Online,NaN,Monitor 27,C143,1,NaN,266.46,entrega rápida
42,1042,NaT,PR,Online,Periféricos,Mouse Gamer,C157,4,NaN,380.26,cliente premium
51,1051,NaT,ES,Loja Física,NaN,Teclado Mecânico,C163,1,NaN,122.47,NaN
53,1053,NaT,SP,Marketplace,Informática,Monitor 27,C150,3,NaN,1387.87,NaN
100,1100,NaT,RJ,Loja Física,Periféricos,Mouse Gamer,C160,4,NaN,324.10,NaN
134,1134,NaT,PR,Marketplace,Informática,Monitor 27,C136,2,NaN,586.71,entrega rápida
138,1138,NaT,RS,Loja Física,Telefonia,Smartphone X,C134,4,NaN,893.48,NaN
144,1144,NaT,MG,Online,Telefonia,Smartphone X,C144,3,NaN,1048.64,entrega rápida


pedido_id       0
data          210
uf              0
canal           0
categoria       0
produto         0
cliente_id      0
quantidade      0
receita         0
lucro           0
observacao      0
dtype: int64


### Registro reflexivo
Escreva aqui sua justificativa para o tratamento dos nulos:
- Em quais colunas você removeu linhas?
- Em quais colunas você preencheu valores?
- Em quais situações decidiu manter o nulo?

1.
Na coluna receita.
Justificativa: Como a receita é a variável principal para calcular o desempenho, linhas sem esse valor (que resultaram em 15 nulos após a tentativa de conversão) foram descartadas para não gerar distorções nos totais.

2.
uf, canal, categoria, lucro e observacao.
Justificativa: Usei "Não Informado" para as categorias para manter o volume de registros. No lucro, usei a mediana (conforme o código anterior) para preencher as lacunas sem enviesar a média por valores extremos.

3.
Na coluna data.
Justificativa: O resultado mostra 210 nulos em data após o pd.to_datetime. Isso indica que o formato original do arquivo é incompatível com a conversão padrão do Pandas. Mantive os nulos temporariamente para não forçar uma informação errada antes de descobrir o formato correto (ex: DD/MM/AAAA vs AAAA-MM-DD).


## 6. Batalha 3 — O ataque dos clones (duplicidades)

A aula alerta que nem toda duplicidade é erro automático: pode haver compra legítima repetida ou dupla inserção do sistema. fileciteturn2file0

### Tarefa
1. Investigue se há linhas duplicadas
2. Analise se a duplicidade parece nociva para os KPIs
3. Escolha uma estratégia:
   - remover duplicidades totais?
   - remover apenas com base em certas colunas?
   - manter?
4. Justifique a decisão


####Estratégia Escolhida:
Remoção baseada na coluna pedido_id.

####Justificativa:
Em sistemas de vendas, o pedido_id deve ser único. Se o mesmo ID aparece duas vezes com os mesmos valores, trata-se de um erro de input ou falha na extração do banco de dados. Manter essas linhas inflaria artificialmente o KPI de Faturamento Total e a Quantidade de Pedidos, levando a conclusões erradas sobre o desempenho da empresa.

####Diferenciação:
Se tivéssemos o mesmo cliente_id comprando o mesmo produto em datas diferentes, seriam vendas legítimas e seriam mantidas. A remoção pelo ID do pedido garante que estamos deletando apenas o erro técnico.

In [ ]:
# Investigue duplicidades e trate o problema conscientemente

# 1.
duplicados_totais = df.duplicated().sum()
print(f"Total de linhas 100% duplicadas: {duplicados_totais}")

# 2.
duplicados_id = df.duplicated(subset=['pedido_id']).sum()
print(f"Total de IDs de pedido repetidos: {duplicados_id}")

# 3.
df.drop_duplicates(subset=['pedido_id'], keep='first', inplace=True)

# 4.
print(f"Tamanho final da base após limpeza de clones: {df.shape}")


Total de linhas 100% duplicadas: 7
Total de IDs de pedido repetidos: 8
Tamanho final da base após limpeza de clones: (205, 11)


## 7. Padronização de categorias

Os slides mostram que categorias duplicadas ou mal escritas podem distorcer rankings e filtros. fileciteturn2file0

### Tarefa
Inspecione e padronize, se necessário:
- `uf`
- `canal`
- `categoria`

### Pense em:
- maiúsculas e minúsculas
- espaços extras
- acentuação / variações
- categorias equivalentes escritas de formas diferentes


###Justificativa:

#### Padronização:
Usei .strip() para remover espaços invisíveis e .upper() para unificar a caixa do texto.

#### Por que isso importa?
Sem isso, o Pandas trataria "Varejo" e "varejo" como duas categorias diferentes em um gráfico de pizza, dividindo o faturamento de forma errada.

#### Variações:
Substituí termos equivalentes (como WEB por ONLINE) para garantir que toda a análise de canais esteja consolidada.

In [ ]:
# Faça aqui a padronização de categorias

# UF, Canal e Categoria
colunas_padronizar = ['uf', 'canal', 'categoria']

for col in colunas_padronizar:
    df[col] = df[col].astype(str).str.strip().str.upper()

# Ajuste de variações
df['canal'] = df['canal'].replace({'MARKET PLACE': 'MARKETPLACE', 'WEB': 'ONLINE'})

# Verificação dos valores únicos
print("Canais padronizados:", df['canal'].unique())
print("Categorias padronizadas:", df['categoria'].unique())


Canais padronizados: ['LOJA FÍSICA' 'NÃO INFORMADO' 'MARKETPLACE' 'ONLINE' 'LOJA FISICA']
Categorias padronizadas: ['INFORMÁTICA' 'PERIFÉRICOS' 'ÁUDIO' 'TELEFONIA' 'NÃO INFORMADO'
 'PERIFERICOS']


## 8. Subindo de nível — Criação de variáveis derivadas

Depois da limpeza, é hora de enriquecer a base com variáveis úteis para análise. fileciteturn2file0

### Tarefa
Crie, se possível:
- `ano`
- `mes`
- `ano_mes`
- `margem_lucro`

### Atenção
A criação de `margem_lucro` exige cuidado com divisões por zero e possíveis valores infinitos.


In [ ]:
# Crie aqui as variáveis derivadas

df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['ano_mes'] = df['data'].dt.to_period('M')

df['margem_lucro'] = (df['lucro'] / df['receita']) * 100
df['margem_lucro'] = df['margem_lucro'].replace([np.inf, -np.inf], 0).fillna(0)

print(df[['ano', 'mes', 'margem_lucro']].head())

      ano  mes  margem_lucro
0  2024.0  4.0     28.440986
1     NaN  NaN     13.515113
2     NaN  NaN     41.505404
3     NaN  NaN     19.815378
4     NaN  NaN     21.488007


### Reflexão técnica
Explique:
1. O que pode acontecer ao calcular margem de lucro quando a receita é zero?
2. Como você decidiu tratar esse caso?

####1.
O Python gera um valor "infinito" ($inf$) ou um erro de divisão por zero. Isso impede cálculos estatísticos posteriores, como a média de margem da empresa.

####2.
Substituí todos os valores infinitos ($inf$) e nulos ($NaN$) resultantes por 0. Isso garante que vendas sem receita não distorçam os indicadores de rentabilidade no dashboard.


## 9. Seleção final — Menos é mais

A aula propõe que nem toda coluna precisa seguir para a base analítica final. fileciteturn2file0

### Tarefa
Crie um DataFrame final, por exemplo `df_clean` ou `df_dash`, contendo apenas as colunas estritamente necessárias para análises de negócio.

**Sugestão de raciocínio:**
- Quais colunas ajudam a responder perguntas de negócio?
- Quais colunas são operacionais, auxiliares ou dispensáveis?
- O que precisa existir para as próximas aulas de visualização e dashboard?


### 1.
Mantive data, uf, canal e categoria para visões de filtros e segmentação. As colunas de receita, lucro e margem_lucro são os indicadores (KPIs) centrais para medir o sucesso das vendas.

### 2.
Removi cliente_id (importante para CRM, mas menos para dashboards de vendas gerais) e observacao, que continha textos irregulares que não geram métricas quantitativas.

### 3.
A base agora tem as dimensões (onde, quando, o quê) e as métricas (quanto) totalmente limpas e padronizadas.

In [ ]:
# Selecione aqui a base final com as colunas escolhidas

colunas_selecionadas = [
    'pedido_id', 'data', 'ano', 'mes', 'uf',
    'canal', 'categoria', 'produto',
    'quantidade', 'receita', 'lucro', 'margem_lucro'
]

df_dash = df[colunas_selecionadas].copy()

print(f"Base final pronta com {df_dash.shape[1]} colunas e {df_dash.shape[0]} linhas.")
display(df_dash.head())


Base final pronta com 12 colunas e 205 linhas.


,pedido_id,data,ano,mes,uf,canal,categoria,produto,quantidade,receita,lucro,margem_lucro
0,1000,2024-04-29,2024.0,4.0,SP,LOJA FÍSICA,INFORMÁTICA,Notebook Pro,1,4535.11,1289.83,28.440986
1,1001,NaT,NaN,NaN,PR,NÃO INFORMADO,INFORMÁTICA,Notebook Pro,1,4005.59,541.36,13.515113
2,1002,NaT,NaN,NaN,PR,MARKETPLACE,PERIFÉRICOS,Teclado Mecânico,1,309.02,128.26,41.505404
3,1003,NaT,NaN,NaN,SP,MARKETPLACE,INFORMÁTICA,Notebook Pro,2,7943.78,1574.09,19.815378
4,1004,NaT,NaN,NaN,RS,ONLINE,INFORMÁTICA,Notebook Pro,5,19926.65,4281.84,21.488007


## 10. Validação final da base limpa

Antes de exportar, faça uma checagem final.

### Verifique:
- os tipos estão corretos?
- ainda há nulos problemáticos?
- ainda há duplicidades nocivas?
- existe algum infinito em `margem_lucro`?
- a base está pronta para ser usada em análises e dashboards?


### 1.
Sim, as métricas estão como float/int e as dimensões como object (ou category). A data está como datetime (mesmo com os nulos que identificamos anteriormente).

### 2.
Fora a coluna data (que precisa de um ajuste de formato específico do arquivo original), as outras colunas críticas de negócio estão 100% preenchidas.

### 3.
Não, o check de pedido_id deve retornar 0.

### 4.
Não, tratamos isso na etapa anterior substituindo por 0.

### 5.
Sim, os dados estão estruturados e limpos para alimentar qualquer ferramenta de BI (Power BI, Tableau ou Looker).

In [ ]:
# Faça aqui a validação final da base limpa

print("1. Tipos de Dados:")
print(df_dash.dtypes)

print("\n2. Contagem de Nulos:")
print(df_dash.isna().sum())

print("\n3. Duplicidades de Pedido ID:")
print(df_dash['pedido_id'].duplicated().sum())

print("\n4. Presença de Infinitos em Margem:")
infinitos = np.isinf(df_dash['margem_lucro']).sum()
print(f"Infinitos encontrados: {infinitos}")

print("\n5. Resumo Estatístico:")
display(df_dash.describe())


1. Tipos de Dados:
pedido_id                int64
data            datetime64[ns]
ano                    float64
mes                    float64
uf                      object
canal                   object
categoria               object
produto                 object
quantidade               int64
receita                float64
lucro                  float64
margem_lucro           float64
dtype: object

2. Contagem de Nulos:
pedido_id         0
data            202
ano             202
mes             202
uf                0
canal             0
categoria         0
produto           0
quantidade        0
receita           0
lucro             0
margem_lucro      0
dtype: int64

3. Duplicidades de Pedido ID:
0

4. Presença de Infinitos em Margem:
Infinitos encontrados: 0

5. Resumo Estatístico:


,pedido_id,data,ano,mes,quantidade,receita,lucro,margem_lucro
count,205.000000,3,3.0,3.000000,205.000000,205.000000,205.000000,205.000000
mean,1109.687805,2024-08-16 16:00:00,2024.0,7.666667,3.019512,4741.736098,1344.775854,34.282582
min,1000.000000,2024-04-29 00:00:00,2024.0,4.000000,1.000000,0.000000,106.780000,0.000000
25%,1057.000000,2024-07-15 00:00:00,2024.0,6.500000,2.000000,871.470000,415.420000,24.364022
50%,1109.000000,2024-09-30 00:00:00,2024.0,9.000000,3.000000,2269.350000,834.010000,33.936033
75%,1166.000000,2024-10-10 12:00:00,2024.0,9.500000,4.000000,7287.540000,1930.280000,42.815863
max,1219.000000,2024-10-21 00:00:00,2024.0,10.000000,6.000000,21240.200000,6335.030000,218.544626
std,63.562692,NaN,0.0,3.214550,1.192135,5075.616384,1328.541930,18.163596


## 11. Exportação da base "Clean"

Agora gere o arquivo final da base tratada.

### Tarefa
Exporte sua base limpa com o nome:

`vendas_brasil_aula3_clean.csv`

### Observação
Esse arquivo será o selo de qualidade do seu trabalho desta aula.


In [ ]:
# Exporte aqui sua base final sem índice

df_dash.to_csv('vendas_brasil_aula3_clean.csv', index=False)

from google.colab import files
files.download('vendas_brasil_aula3_clean.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Conclusão e registro reflexivo

Escreva um pequeno texto respondendo:

1. Quais foram os principais problemas de qualidade encontrados?
2. Qual decisão de limpeza foi mais difícil?
3. Que impacto essas falhas poderiam causar em um dashboard?
4. Por que a etapa de limpeza é considerada a "fundação invisível" do projeto?


###1
Principais problemas: Tipos incorretos (data/receita como texto), valores nulos em colunas críticas (lucro/receita), duplicidade de IDs e falta de padronização (maiúsculas/minúsculas).

###2
A exclusão de linhas com receita nula. É um dilema entre perder volume de dados ou manter registros que distorceriam o faturamento real.

###3
Dashboards com valores inflados (duplicados), gráficos de linha quebrados (datas como texto) e filtros que não funcionam por erro de digitação.

###4
Porque o visual só é confiável se a base estiver íntegra. Se a limpeza falha, qualquer decisão baseada no gráfico será um erro estratégico.

## 13. Desafio extra (opcional)

Use sua base limpa para responder rapidamente:

- Qual UF concentra maior receita?
- Qual canal gera maior lucro?
- Existe algum mês com desempenho claramente acima dos demais?

Você pode responder com tabelas simples ou pequenos agrupamentos.


In [ ]:
# Desafio extra opcional

# 1.
print("Receita por UF:")
print(df_dash.groupby('uf')['receita'].sum().sort_values(ascending=False).head(3))

# 2.
print("\nLucro por Canal:")
print(df_dash.groupby('canal')['lucro'].sum().sort_values(ascending=False))

# 3.
print("\nDesempenho por Mês (Lucro):")
print(df_dash.groupby('mes')['lucro'].sum().sort_values(ascending=False))


Receita por UF:
uf
SC    201535.66
ES    160311.74
MG    152331.56
Name: receita, dtype: float64

Lucro por Canal:
canal
ONLINE           91184.40
MARKETPLACE      90753.11
LOJA FÍSICA      86996.80
NÃO INFORMADO     5283.85
LOJA FISICA       1460.89
Name: lucro, dtype: float64

Desempenho por Mês (Lucro):
mes
10.0    2087.05
4.0     1289.83
9.0      641.54
Name: lucro, dtype: float64
